In [3]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

base = "/hkfs/work/workspace/scratch/xo8179-nextgems_regridded/"

ds = xr.open_dataset(base+"3D_nextgems_1990s_6hourly_128x64_z_no_nans.zarr", chunks={"time": 16})

In [4]:
def plot_num_nans(dataset):
    for var in dataset.variables:
        total_nans = 0
        if "time" in dataset[var].dims:
            for i in range(1990, 2020):
                total_nans = total_nans + np.isnan(dataset[var].sel(time=f'{i}').values.flatten()).sum()
            print(f"Number of NaNs for {var}: {total_nans}")
        else:
            print(f"Number of NaNs for {var}: {np.isnan(dataset[var].values.flatten()).sum()}")

In [5]:
print(ds)

<xarray.Dataset> Size: 12GB
Dimensions:  (lat: 64, level: 13, lon: 128, time: 14608)
Coordinates:
  * lat      (lat) float64 512B -90.0 -87.14 -84.29 -81.43 ... 84.29 87.14 90.0
  * level    (level) int64 104B 50 100 150 200 250 300 ... 600 700 850 925 1000
  * lon      (lon) float64 1kB 0.0 2.812 5.625 8.438 ... 348.8 351.6 354.4 357.2
  * time     (time) datetime64[ns] 117kB 1990-01-01 ... 1999-12-31T18:00:00
Data variables:
    z        (time, level, lon, lat) float64 12GB dask.array<chunksize=(16, 13, 128, 64), meta=np.ndarray>


In [7]:

        
#plot_num_nans(ds)
var = "z"
nan_locations = []

for year in range(1990, 2000):
    da = ds[var].sel(time=str(year))
    mask = np.isnan(da)

    if mask.any():
        # Get indexes of NaNs
        idxs = np.argwhere(mask.values)
        for idx in idxs:
            # Extract coordinates for each NaN
            coords = {}
            for axis, dim in enumerate(da.dims):
                coord_val = da[dim].values[idx[axis]]
                coords[dim] = idx
            coords["year"] = year
            nan_locations.append(coords)

# Convert to DataFrame for easy viewing
df_nan_locations = pd.DataFrame(nan_locations)
print(df_nan_locations)

Empty DataFrame
Columns: []
Index: []
